In [ ]:
from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")

transformer_paths = list(input_root.rglob("transformer_holdout.csv"))
if not transformer_paths:
    raise FileNotFoundError("Attach the successful Transformer notebook output.")

transformer_path = transformer_paths[0]
score_dir = transformer_path.parent
if_path = score_dir / "isolation_forest.csv"

label_paths = [
    p for p in input_root.rglob("labels.csv")
    if (p.parent / "manifest.csv").is_file()
]
if not if_path.is_file():
    raise FileNotFoundError("Isolation Forest scores are missing from the attached output.")
if not label_paths:
    raise FileNotFoundError("Attach the original 5,000-file notebook output for catalog labels.")

transformer = pd.read_csv(transformer_path)
isolation = pd.read_csv(if_path)
labels = pd.read_csv(label_paths[0])

keys = ["source_id", "observation_id", "sector"]

def normalize_ids(frame, columns):
    frame = frame.copy()
    for column in columns:
        frame[column] = frame[column].astype("string").str.replace(
            r"\.0$", "", regex=True
        )
    return frame

transformer = normalize_ids(transformer, keys)
isolation = normalize_ids(isolation, keys)
labels = normalize_ids(labels, ["source_id"])

joined = transformer.merge(
    isolation[keys + ["isolation_forest_anomaly_score"]],
    on=keys,
    how="left",
    validate="one_to_one",
)

# Compare both methods on the same held-out observations.
joined["isolation_forest_holdout_percentile"] = (
    joined["isolation_forest_anomaly_score"].rank(pct=True)
)
joined["consensus_percentile"] = joined[
    ["transformer_percentile", "isolation_forest_holdout_percentile"]
].mean(axis=1)

joined = joined.merge(
    labels,
    on="source_id",
    how="left",
    validate="many_to_one",
)

# Keep each source's highest-ranked observation.
candidates = (
    joined.sort_values("consensus_percentile", ascending=False)
    .drop_duplicates("source_id")
)

output = Path("/kaggle/working/candidate_consensus.csv")
candidates.to_csv(output, index=False)

print("Transformer observations:", len(transformer))
print("Matched Isolation Forest scores:", joined["isolation_forest_anomaly_score"].notna().sum())
print("Saved candidate table:", output)
display(candidates[
    ["source_id", "observation_id", "sector", "consensus_percentile",
     "transformer_percentile", "isolation_forest_holdout_percentile",
     "class_label", "evidence_level", "gaia_class_name",
     "tess_eb_catalog_match"]
].head(25))

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

raw_dir = Path(label_paths[0]).parent / "raw"
top = candidates.head(12).reset_index(drop=True)

fig, axes = plt.subplots(4, 3, figsize=(16, 12))
axes = axes.flatten()

for ax, (_, row) in zip(axes, top.iterrows()):
    fits_path = raw_dir / Path(str(row["path"])).name

    if not fits_path.is_file():
        ax.set_title(f'{row["source_id"]}: FITS file not found')
        continue

    with fits.open(fits_path, memmap=True) as hdul:
        data = hdul[1].data
        columns = {name.upper(): name for name in data.names}

        time = np.asarray(data[columns["TIME"]], dtype=float)
        flux_name = columns.get("PDCSAP_FLUX", columns.get("SAP_FLUX"))
        flux = np.asarray(data[flux_name], dtype=float)

        finite = np.isfinite(time) & np.isfinite(flux)
        if "QUALITY" in columns:
            quality = np.asarray(data[columns["QUALITY"]], dtype=float)
            good = finite & (quality == 0)
            if good.sum() < 32:
                good = finite
        else:
            good = finite

        normalized_flux = flux[good] / np.nanmedian(flux[good])
        ax.scatter(time[good], normalized_flux, s=1, alpha=0.6)

    label = row.get("evidence_level", "no catalog match")
    ax.set_title(
        f'TIC {row["source_id"]} | sector {row["sector"]} '
        f'| rank {row["consensus_percentile"]:.3f} | {label}'
    )
    ax.set_xlabel("Time (BTJD)")
    ax.set_ylabel("Normalized flux")

for ax in axes[len(top):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.timeseries import BoxLeastSquares

raw_dir = label_paths[0].parent / "raw"
targets = [229137615, 30317282, 307364493]
durations = np.array([0.03, 0.06, 0.12, 0.24])  # days

fig, axes = plt.subplots(len(targets), 2, figsize=(13, 11))

for row_index, tic in enumerate(targets):
    match = candidates[candidates["source_id"].astype(str) == str(tic)]
    if match.empty:
        print(f"TIC {tic}: not found in the candidate table")
        continue

    candidate = match.iloc[0]
    fits_path = raw_dir / Path(str(candidate["path"])).name

    if not fits_path.is_file():
        print(f"TIC {tic}: FITS file not found: {fits_path}")
        continue

    with fits.open(fits_path, memmap=True) as hdul:
        data = hdul[1].data
        columns = {name.upper(): name for name in data.names}

        time = np.asarray(data[columns["TIME"]], dtype=float)
        flux_name = columns.get("PDCSAP_FLUX", columns.get("SAP_FLUX"))
        flux = np.asarray(data[flux_name], dtype=float)

        good = np.isfinite(time) & np.isfinite(flux)
        if "QUALITY" in columns:
            quality = np.asarray(data[columns["QUALITY"]], dtype=float)
            quality_good = good & (quality == 0)
            if quality_good.sum() >= 32:
                good = quality_good

    time, flux = time[good], flux[good]
    normalized_flux = flux / np.nanmedian(flux)

    max_period = min(10.0, (time.max() - time.min()) / 2)
    if max_period <= 0.4:
        print(f"TIC {tic}: not enough time coverage for this period search")
        continue

    bls = BoxLeastSquares(time, normalized_flux)
    periodogram = bls.autopower(
        duration=durations,
        minimum_period=0.4,
        maximum_period=max_period,
    )

    best = np.argmax(periodogram.power)
    period = periodogram.period[best]
    transit_time = periodogram.transit_time[best]
    depth = periodogram.depth[best]
    phase = ((time - transit_time + period / 2) % period) / period - 0.5

    axes[row_index, 0].scatter(time, normalized_flux, s=1, alpha=0.5)
    axes[row_index, 0].set_title(f"TIC {tic}: original light curve")
    axes[row_index, 0].set_xlabel("Time (BTJD)")
    axes[row_index, 0].set_ylabel("Normalized flux")

    axes[row_index, 1].scatter(phase, normalized_flux, s=1, alpha=0.5)
    axes[row_index, 1].set_title(f"Folded at {period:.4f} days")
    axes[row_index, 1].set_xlabel("Phase")
    axes[row_index, 1].set_ylabel("Normalized flux")

    print(f"TIC {tic}: best trial period={period:.4f} days; depth={depth:.5f}")

plt.tight_layout()
plt.show()

In [ ]:
from astropy.timeseries import BoxLeastSquares
from pathlib import Path
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

raw_dir = label_paths[0].parent / "raw"
targets = [229137615, 30317282, 307364493]
durations = np.array([0.03, 0.06, 0.12, 0.24])  # days

fig, axes = plt.subplots(len(targets), 2, figsize=(13, 11))

for i, tic in enumerate(targets):
    match = candidates[candidates["source_id"].astype(str) == str(tic)]
    if match.empty:
        print(f"TIC {tic}: not found in candidates")
        continue

    row = match.iloc[0]
    fits_path = raw_dir / Path(str(row["path"])).name

    with fits.open(fits_path, memmap=True) as hdul:
        data = hdul[1].data
        cols = {name.upper(): name for name in data.names}
        time = np.asarray(data[cols["TIME"]], dtype=float)
        flux_name = cols.get("PDCSAP_FLUX", cols.get("SAP_FLUX"))
        flux = np.asarray(data[flux_name], dtype=float)

        good = np.isfinite(time) & np.isfinite(flux)
        if "QUALITY" in cols:
            quality = np.asarray(data[cols["QUALITY"]], dtype=float)
            quality_good = good & (quality == 0)
            if quality_good.sum() >= 32:
                good = quality_good

    time, flux = time[good], flux[good]
    flux = flux / np.nanmedian(flux)
    max_period = min(10.0, (time.max() - time.min()) / 2)

    bls = BoxLeastSquares(time, flux)
    result = bls.autopower(
        duration=durations,
        minimum_period=0.4,
        maximum_period=max_period,
    )

    best = np.argmax(result.power)
    period = result.period[best]
    t0 = result.transit_time[best]
    phase = ((time - t0 + period / 2) % period) / period - 0.5

    axes[i, 0].scatter(time, flux, s=1)
    axes[i, 0].set_title(f"TIC {tic}: original curve")
    axes[i, 0].set_xlabel("Time (BTJD)")
    axes[i, 0].set_ylabel("Normalized flux")

    axes[i, 1].scatter(phase, flux, s=1)
    axes[i, 1].set_title(f"Folded at {period:.4f} days")
    axes[i, 1].set_xlabel("Phase")
    axes[i, 1].set_ylabel("Normalized flux")

    print(f"TIC {tic}: trial period {period:.4f} days")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.timeseries import BoxLeastSquares

# Reuse the candidate table from the earlier cells.
# If it isn't in memory, load the saved table.
if "candidates" not in globals():
    candidates = pd.read_csv("/kaggle/working/candidate_consensus.csv")

# Rank candidates, then skip the three already checked.
rank_col = "consensus_percentile" if "consensus_percentile" in candidates.columns else "candidate_score_v1"
ranked = candidates.sort_values(rank_col, ascending=False).copy()

already_checked = {"229137615", "30317282", "307364493"}
ranked["_tic"] = ranked["source_id"].astype(str).str.replace(r"\.0$", "", regex=True)
next_batch = ranked.loc[~ranked["_tic"].isin(already_checked)].head(17)

if "path" not in next_batch.columns:
    raise KeyError(f"No FITS path column. Available columns: {list(next_batch.columns)}")

raw_dir = Path("/kaggle/working/cbs-data/raw")
if "raw_dir" in globals():
    raw_dir = Path(raw_dir)

results = []
for _, row in next_batch.iterrows():
    fits_path = Path(str(row["path"]))
    if not fits_path.is_file():
        fits_path = raw_dir / fits_path.name
    if not fits_path.is_file():
        print(f"Skipping TIC {row['_tic']}: FITS file not found")
        continue

    try:
        with fits.open(fits_path, memmap=False) as hdul:
            data = hdul[1].data
            names = {name.upper(): name for name in data.names}
            flux_col = names.get("PDCSAP_FLUX", names.get("SAP_FLUX"))
            if flux_col is None or "TIME" not in names:
                print(f"Skipping TIC {row['_tic']}: TIME/flux columns missing")
                continue

            time = np.asarray(data[names["TIME"]], dtype=float)
            flux = np.asarray(data[flux_col], dtype=float)
            valid = np.isfinite(time) & np.isfinite(flux) & (flux > 0)

            if "QUALITY" in names:
                valid &= np.asarray(data[names["QUALITY"]]) == 0

            time, flux = time[valid], flux[valid]
            if len(time) < 200 or np.ptp(time) < 1:
                print(f"Skipping TIC {row['_tic']}: too few usable points")
                continue

            flux = flux / np.median(flux)
            max_period = min(10.0, np.ptp(time) / 2.0)
            if max_period <= 0.4:
                continue

            search = BoxLeastSquares(time, flux).autopower(
                duration=[0.04, 0.08, 0.16, 0.32],
                minimum_period=0.4,
                maximum_period=max_period,
                objective="snr",
                frequency_factor=5,
            )
            best = int(np.nanargmax(search.power))

            results.append({
                "source_id": row["_tic"],
                "sector": row.get("sector", np.nan),
                "consensus_percentile": row.get("consensus_percentile", np.nan),
                "trial_period_days": float(search.period[best]),
                "depth": float(search.depth[best]),
                "duration_days": float(search.duration[best]),
                "bls_power": float(search.power[best]),
                "fits_file": fits_path.name,
            })
            print(f"TIC {row['_tic']}: trial period {search.period[best]:.4f} days")

    except Exception as e:
        print(f"Skipping TIC {row['_tic']}: {type(e).__name__}: {e}")

results_df = pd.DataFrame(results)
out_file = raw_dir.parent / "scores" / "next_candidates_bls.csv"
out_file.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_file, index=False)

print(f"\nSaved {len(results_df)} results to: {out_file}")
display(results_df.sort_values("consensus_percentile", ascending=False))

In [ ]:
from pathlib import Path

print("Candidate columns:", list(candidates.columns))
if "path" in candidates.columns:
    print("\nExample candidate paths:")
    print(candidates[["source_id", "path"]].head(3).to_string(index=False))

print("\nCurrent raw_dir:", globals().get("raw_dir"))
print("\nSaved manifests and their raw folders:")
for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if root.exists():
        for manifest in root.rglob("manifest.csv"):
            raw_folder = manifest.parent / "raw"
            print(manifest, "| raw exists:", raw_folder.is_dir())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.timeseries import BoxLeastSquares

# Reuse the ranked candidate table already loaded in the notebook.
if "candidates" not in globals():
    candidates = pd.read_csv("/kaggle/working/candidate_consensus.csv")

raw_dir = Path(
    "/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data/raw"
)

rank_col = (
    "consensus_percentile"
    if "consensus_percentile" in candidates.columns
    else "candidate_score_v1"
)

ranked = candidates.sort_values(rank_col, ascending=False).copy()
ranked["_tic"] = ranked["source_id"].astype(str).str.replace(r"\.0$", "", regex=True)

already_checked = {"229137615", "30317282", "307364493"}
next_batch = ranked.loc[~ranked["_tic"].isin(already_checked)].head(17)

results = []

for _, row in next_batch.iterrows():
    old_path = Path(str(row["path"]))
    fits_path = old_path if old_path.is_file() else raw_dir / old_path.name

    if not fits_path.is_file():
        print(f"Skipping TIC {row['_tic']}: not found at {fits_path}")
        continue

    try:
        with fits.open(fits_path, memmap=False) as hdul:
            data = hdul[1].data
            names = {name.upper(): name for name in data.names}

            flux_col = names.get("PDCSAP_FLUX", names.get("SAP_FLUX"))
            if "TIME" not in names or flux_col is None:
                print(f"Skipping TIC {row['_tic']}: TIME/flux columns missing")
                continue

            time = np.asarray(data[names["TIME"]], dtype=float)
            flux = np.asarray(data[flux_col], dtype=float)
            valid = np.isfinite(time) & np.isfinite(flux) & (flux > 0)

            if "QUALITY" in names:
                valid &= np.asarray(data[names["QUALITY"]]) == 0

            time, flux = time[valid], flux[valid]

            if len(time) < 200 or np.ptp(time) < 1:
                print(f"Skipping TIC {row['_tic']}: too few usable points")
                continue

            flux = flux / np.median(flux)
            max_period = min(10.0, np.ptp(time) / 2.0)

            if max_period <= 0.4:
                print(f"Skipping TIC {row['_tic']}: time baseline too short")
                continue

            search = BoxLeastSquares(time, flux).autopower(
                duration=[0.04, 0.08, 0.16, 0.32],
                minimum_period=0.4,
                maximum_period=max_period,
                objective="snr",
                frequency_factor=5,
            )

            best = int(np.nanargmax(search.power))
            period = float(search.period[best])

            results.append({
                "source_id": row["_tic"],
                "sector": row.get("sector", np.nan),
                "consensus_percentile": row.get("consensus_percentile", np.nan),
                "trial_period_days": period,
                "depth": float(search.depth[best]),
                "duration_days": float(search.duration[best]),
                "bls_power": float(search.power[best]),
                "fits_file": fits_path.name,
            })

            print(f"TIC {row['_tic']}: trial period {period:.4f} days")

    except Exception as e:
        print(f"Skipping TIC {row['_tic']}: {type(e).__name__}: {e}")

results_df = pd.DataFrame(results)
out_file = Path("/kaggle/working/next_candidates_bls.csv")
results_df.to_csv(out_file, index=False)

print(f"\nSaved {len(results_df)} results to: {out_file}")

if results_df.empty:
    print("No results were produced; check the messages above.")
else:
    display(results_df.sort_values("consensus_percentile", ascending=False))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Load the ranked table and the 17 period-search results.
candidate_file = Path("/kaggle/working/candidate_consensus.csv")
candidates = (
    candidates.copy()
    if "candidates" in globals()
    else pd.read_csv(candidate_file)
)
bls = pd.read_csv("/kaggle/working/next_candidates_bls.csv")

# Add the three candidates already period-folded.
prior_periods = pd.DataFrame([
    {"source_id": 229137615, "sector": 2,  "trial_period_days": 9.6835},
    {"source_id": 30317282,  "sector": 10, "trial_period_days": 1.3977},
    {"source_id": 307364493, "sector": 5,  "trial_period_days": 8.6470},
])
prior_periods["period_search"] = "previous BLS fold"
bls["period_search"] = "BLS batch"

periods = pd.concat([bls, prior_periods], ignore_index=True, sort=False)

# Normalize IDs and sectors so the two tables merge reliably.
def normalize_id(values):
    return values.astype(str).str.replace(r"\.0$", "", regex=True)

candidates["_tic"] = normalize_id(candidates["source_id"])
periods["_tic"] = normalize_id(periods["source_id"])
candidates["_sector"] = pd.to_numeric(candidates["sector"], errors="coerce").astype("Int64")
periods["_sector"] = pd.to_numeric(periods["sector"], errors="coerce").astype("Int64")

rank_col = "consensus_percentile" if "consensus_percentile" in candidates.columns else "candidate_score_v1"
top20 = (
    candidates.sort_values(rank_col, ascending=False)
    .drop_duplicates(["_tic", "_sector"])
    .head(20)
)

review = top20.merge(
    periods.drop_duplicates(["_tic", "_sector"], keep="last"),
    on=["_tic", "_sector"],
    how="left",
    suffixes=("", "_period"),
)

eb_match = (
    review["tess_eb_catalog_match"].astype(str).str.lower().isin(["true", "1", "yes"])
    if "tess_eb_catalog_match" in review.columns
    else pd.Series(False, index=review.index)
)
has_label = (
    review["evidence_level"].astype(str).str.lower().isin(["single_catalog", "well_supported"])
    if "evidence_level" in review.columns
    else pd.Series(False, index=review.index)
)
duration_at_search_limit = (
    pd.to_numeric(review["duration_days"], errors="coerce") >= 0.319
    if "duration_days" in review.columns
    else pd.Series(False, index=review.index)
)

review["review_note"] = np.select(
    [
        eb_match,
        has_label,
        duration_at_search_limit,
        review["trial_period_days"].notna(),
    ],
    [
        "TESS EB catalog match; use as a known-variable control",
        "Catalog-labeled variable; review its folded curve",
        "BLS chose the 0.32-day search limit; period is provisional",
        "Period is provisional; inspect the folded curve",
    ],
    default="No period result; inspect the light curve",
)

output_columns = [
    c for c in [
        "source_id", "sector", "consensus_percentile",
        "trial_period_days", "depth", "duration_days", "bls_power",
        "evidence_level", "class_label", "gaia_class_name",
        "tess_eb_catalog_match", "review_note",
    ]
    if c in review.columns
]

out_file = Path("/kaggle/working/cosmic_swan_candidate_review.csv")
review[output_columns].to_csv(out_file, index=False)

print(f"Saved review table: {out_file}")
print(f"Candidates reviewed: {len(review)}")
print(f"TESS EB catalog matches: {int(eb_match.sum())}")
print(f"BLS durations at search limit: {int(duration_at_search_limit.sum())}")
display(review[output_columns])

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

review = pd.read_csv("/kaggle/working/cosmic_swan_candidate_review.csv")

def truthy(series):
    return series.astype(str).str.lower().isin({"true", "1", "yes"})

eb_match = truthy(review["tess_eb_catalog_match"])
accepted_evidence = (
    review["evidence_level"].astype(str).str.lower()
    .isin({"single_catalog", "well_supported"})
)
gaia_class = (
    review["gaia_class_name"].notna()
    & ~review["gaia_class_name"].astype(str).str.lower().isin({"", "none", "none_found", "nan"})
)
has_catalog_info = eb_match | accepted_evidence | gaia_class
duration = pd.to_numeric(review["duration_days"], errors="coerce")
period = pd.to_numeric(review["trial_period_days"], errors="coerce")

review["bls_duration_at_search_limit"] = duration >= 0.319
review["final_vetting_status"] = np.select(
    [
        eb_match,
        accepted_evidence | gaia_class,
        review["bls_duration_at_search_limit"],
        period.notna(),
    ],
    [
        "TESS eclipsing-binary catalog match",
        "Catalog-classified variable; inspect manually",
        "Broad BLS fit; trial period uncertain",
        "Unmatched provisional period; inspect folded curve",
    ],
    default="Not yet period-vetted",
)

# Unmatched means no match in the catalogs used; it does not mean newly discovered.
review["catalog_unmatched"] = ~has_catalog_info

review = review.sort_values("consensus_percentile", ascending=False)
review_file = Path("/kaggle/working/cosmic_swan_final_review.csv")
review.to_csv(review_file, index=False)

follow_up = review.loc[review["catalog_unmatched"]].head(10)
follow_up_file = Path("/kaggle/working/cosmic_swan_follow_up_shortlist.csv")
follow_up.to_csv(follow_up_file, index=False)

report_file = Path("/kaggle/working/cosmic_swan_screening_summary.md")
report_file.write_text(
    "\n".join([
        "# Cosmic Black Swan: screening summary",
        "",
        f"- Candidates reviewed: {len(review)}",
        f"- TESS eclipsing-binary catalog matches: {int(eb_match.sum())}",
        f"- Candidates with catalog classification/evidence: {int(has_catalog_info.sum())}",
        f"- BLS fits at the maximum tested duration: {int((duration >= 0.319).sum())}",
        f"- Candidates in the unmatched follow-up queue: {len(follow_up)}",
        "",
        "BLS periods are trial values, not confirmed orbital periods. "
        "A catalog-unmatched object is not thereby a new discovery. "
        "The follow-up shortlist requires inspection of the light curves and "
        "independent validation before any transient claim.",
        "",
        f"Full review table: `{review_file}`",
        f"Unmatched follow-up shortlist: `{follow_up_file}`",
    ]),
    encoding="utf-8",
)

print(f"Saved: {review_file}")
print(f"Saved: {follow_up_file}")
print(f"Saved: {report_file}")
display(follow_up)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from IPython.display import display

shortlist = pd.read_csv("/kaggle/working/cosmic_swan_follow_up_shortlist.csv")

# Reuse the ranked table that contains the FITS paths.
if "candidates" not in globals() or "path" not in candidates.columns:
    candidates = pd.read_csv("/kaggle/working/candidate_consensus.csv")

def clean_id(series):
    return series.astype(str).str.replace(r"\.0$", "", regex=True)

shortlist["_tic"] = clean_id(shortlist["source_id"])
candidates["_tic"] = clean_id(candidates["source_id"])
shortlist["_sector"] = pd.to_numeric(shortlist["sector"], errors="coerce").astype("Int64")
candidates["_sector"] = pd.to_numeric(candidates["sector"], errors="coerce").astype("Int64")

path_lookup = candidates[["_tic", "_sector", "path"]].drop_duplicates(
    ["_tic", "_sector"]
)
shortlist = shortlist.merge(path_lookup, on=["_tic", "_sector"], how="left")

raw_dir = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data/raw")
plots = []

for _, row in shortlist.iterrows():
    period = pd.to_numeric(row.get("trial_period_days"), errors="coerce")
    if not np.isfinite(period) or period <= 0 or pd.isna(row.get("path")):
        print(f"Skipping TIC {row['_tic']}: missing FITS path or trial period")
        continue

    old_path = Path(str(row["path"]))
    fits_path = old_path if old_path.is_file() else raw_dir / old_path.name

    if not fits_path.is_file():
        print(f"Skipping TIC {row['_tic']}: FITS file not found: {fits_path}")
        continue

    try:
        with fits.open(fits_path, memmap=False) as hdul:
            data = hdul[1].data
            names = {name.upper(): name for name in data.names}
            flux_col = names.get("PDCSAP_FLUX", names.get("SAP_FLUX"))

            if "TIME" not in names or flux_col is None:
                print(f"Skipping TIC {row['_tic']}: TIME/flux columns missing")
                continue

            time = np.asarray(data[names["TIME"]], dtype=float)
            flux = np.asarray(data[flux_col], dtype=float)
            valid = np.isfinite(time) & np.isfinite(flux) & (flux > 0)

            if "QUALITY" in names:
                valid &= np.asarray(data[names["QUALITY"]]) == 0

            time, flux = time[valid], flux[valid]
            if len(time) < 100:
                print(f"Skipping TIC {row['_tic']}: too few usable points")
                continue

            flux /= np.median(flux)
            order = np.argsort(time)
            time, flux = time[order], flux[order]

            # Thin only for plotting; the period search result is unchanged.
            step = max(1, len(time) // 12000)
            t_plot, f_plot = time[::step], flux[::step]
            phase = ((t_plot - t_plot.min() + period / 2) % period) / period - 0.5

            plots.append((row, period, t_plot, f_plot, phase))

    except Exception as e:
        print(f"Skipping TIC {row['_tic']}: {type(e).__name__}: {e}")

if not plots:
    print("No curves were plotted; check the file messages above.")
else:
    fig, axes = plt.subplots(len(plots), 2, figsize=(14, 3.2 * len(plots)), squeeze=False)

    for i, (row, period, time, flux, phase) in enumerate(plots):
        ax_time, ax_phase = axes[i]

        ax_time.scatter(time, flux, s=1)
        ax_time.set_title(f"TIC {row['_tic']} | original curve")
        ax_time.set_xlabel("Time (BTJD)")
        ax_time.set_ylabel("Normalized flux")

        ax_phase.scatter(phase, flux, s=1)
        ax_phase.set_title(f"Folded at trial period {period:.4f} days")
        ax_phase.set_xlabel("Phase")
        ax_phase.set_ylabel("Normalized flux")

    fig.tight_layout()
    plot_file = Path("/kaggle/working/cosmic_swan_follow_up_vetting.png")
    fig.savefig(plot_file, dpi=160, bbox_inches="tight")
    print(f"Saved plot: {plot_file}")
    display(fig)

In [ ]:
import pandas as pd
from pathlib import Path

shortlist = pd.read_csv("/kaggle/working/cosmic_swan_follow_up_shortlist.csv")

visual_notes = {
    "229137615": "Repeated broad dips; folded curve shows recurring structure.",
    "4637806": "Noisy low-amplitude variability; no clear isolated event visible.",
    "11521787": "Oscillatory variability; the trial period remains uncertain.",
    "19566581": "Noisy variation; no clear isolated event visible.",
    "1008059": "Broad dips recur; trial period may be an alias.",
    "21165132": "Noisy variability; folded curve lacks a clear isolated event.",
    "300039724": "Low-amplitude variation; no clear isolated event visible.",
    "11382822": "Broad wave-like changes recur in the light curve.",
    "5370150": "Broad cyclic variation; BLS period is uncertain.",
    "1085150": "Repeated dips; folded curve shows recurring structure.",
}

shortlist["source_id"] = (
    shortlist["source_id"].astype(str).str.replace(r"\.0$", "", regex=True)
)
shortlist["visual_review_note"] = shortlist["source_id"].map(visual_notes)
shortlist["visual_review_status"] = "No confirmed isolated event in displayed curve"
shortlist["review_limit"] = (
    "Visual triage of one TESS sector; catalog-unmatched does not establish novelty."
)

out_file = Path("/kaggle/working/cosmic_swan_follow_up_visual_review.csv")
shortlist.to_csv(out_file, index=False)

print(f"Saved visual review: {out_file}")
display(shortlist[[
    "source_id",
    "sector",
    "consensus_percentile",
    "trial_period_days",
    "visual_review_note",
    "visual_review_status",
]])

In [ ]:
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "astroquery"],
    check=True,
)

from cosmic_black_swan.pixel_audit import run_pixel_audit

result = run_pixel_audit(
    data_root,
    source_id="229137615",
    sector=2,
)
print(result)

In [ ]:
import sys
import shutil
from pathlib import Path

source_root = Path(
    "/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-project-src/src"
)
working_root = Path("/kaggle/working/cbs-project-src/src")
shutil.copytree(source_root, working_root, dirs_exist_ok=True)

audit_file = working_root / "cosmic_black_swan" / "pixel_audit.py"
source = audit_file.read_text(encoding="utf-8")

old = '''observations = Observations.query_region(center, radius=3.0 * u.arcsec, obs_collection="TESS")
    if observations is None or len(observations) == 0:
        raise RuntimeError("MAST returned no nearby TESS observations")'''

new = '''observations = Observations.query_region(center, radius=3.0 * u.arcsec)
    if observations is not None and "obs_collection" in observations.colnames:
        observations = observations[observations["obs_collection"] == "TESS"]
    if observations is None or len(observations) == 0:
        raise RuntimeError("MAST returned no nearby TESS observations")'''

if old not in source:
    raise RuntimeError("Expected query code not found; stop and show me this output.")

audit_file.write_text(source.replace(old, new, 1), encoding="utf-8")

# Load the patched project copy instead of the read-only Kaggle input copy.
sys.path.insert(0, str(working_root))
for name in list(sys.modules):
    if name == "cosmic_black_swan" or name.startswith("cosmic_black_swan."):
        del sys.modules[name]

from cosmic_black_swan.pixel_audit import run_pixel_audit
import json

result = run_pixel_audit(data_root, source_id="229137615", sector=2)
print(json.dumps(result, indent=2))

In [ ]:
import json
from pathlib import Path

report_path = Path("/kaggle/working/cbs-data/pixel_audits/TIC_229137615.json")
print(json.dumps(json.loads(report_path.read_text()), indent=2))

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
from astropy.io import fits

project_src = Path("/kaggle/working/cbs-project-src/src")
audit_file = project_src / "cosmic_black_swan" / "pixel_audit.py"

source = audit_file.read_text(encoding="utf-8")
old = """x, y = wcs.world_to_pixel_values(float(ra), float(dec))
                x_near, y_near = int(round(x)), int(round(y))"""
new = """x, y = wcs.world_to_pixel_values(float(ra), float(dec))
                x = float(np.asarray(x).squeeze())
                y = float(np.asarray(y).squeeze())
                x_near, y_near = int(round(x)), int(round(y))"""

if old not in source:
    raise RuntimeError("Expected Gaia pixel-conversion code not found; stop and show me this output.")
audit_file.write_text(source.replace(old, new, 1), encoding="utf-8")

sys.path.insert(0, str(project_src))
for name in list(sys.modules):
    if name == "cosmic_black_swan" or name.startswith("cosmic_black_swan."):
        del sys.modules[name]

from cosmic_black_swan.pixel_audit import _audit_one

report_path = Path("/kaggle/working/cbs-data/pixel_audits/TIC_229137615.json")
report = json.loads(report_path.read_text())
product = report["target_pixel_products"][0]
fits_path = Path(product["file"])

with fits.open(fits_path, memmap=True) as hdul:
    aperture = np.asarray(hdul[2].data)
    values, counts = np.unique(aperture, return_counts=True)
    print("Aperture mask shape:", aperture.shape)
    print("Aperture mask values/counts:", dict(zip(values.tolist(), counts.tolist())))

audit = _audit_one(fits_path, "229137615", report["gaia_query"])
print("Selected aperture pixels:", audit["optimal_aperture_pixels"])
print("Gaia neighbor positions:")
print(json.dumps(audit["gaia_neighbors_within_30_arcsec"], indent=2))

In [ ]:
import sys, json
from pathlib import Path

project_src = Path("/kaggle/working/cbs-project-src/src")
audit_file = project_src / "cosmic_black_swan" / "pixel_audit.py"

source = audit_file.read_text(encoding="utf-8")
old = "optimal_mask = (aperture.astype(np.int64) & 1) > 0"
new = "optimal_mask = (aperture.astype(np.int64) & 2) > 0"

if old not in source:
    raise RuntimeError("Expected aperture-mask line not found; stop and show me this output.")
audit_file.write_text(source.replace(old, new, 1), encoding="utf-8")

sys.path.insert(0, str(project_src))
for name in list(sys.modules):
    if name == "cosmic_black_swan" or name.startswith("cosmic_black_swan."):
        del sys.modules[name]

from cosmic_black_swan.pixel_audit import _audit_one

report_path = Path("/kaggle/working/cbs-data/pixel_audits/TIC_229137615.json")
report = json.loads(report_path.read_text())
product = report["target_pixel_products"][0]

audit = _audit_one(
    Path(product["file"]),
    "229137615",
    report["gaia_query"],
)
report["target_pixel_products"][0] = audit
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Corrected report saved:", report_path)
print(json.dumps({
    "optimal_aperture_pixels": audit["optimal_aperture_pixels"],
    "aperture_flux_fraction": audit["aperture_positive_flux_fraction_of_cutout"],
    "outside_aperture_flux_fraction": audit["outside_aperture_positive_flux_fraction_of_cutout"],
    "gaia_neighbors": audit["gaia_neighbors_within_30_arcsec"],
    "bright_neighbors_in_aperture": audit["bright_gaia_neighbors_in_optimal_aperture"],
}, indent=2))

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astropy.io import fits

report_path = Path("/kaggle/working/cbs-data/pixel_audits/TIC_229137615.json")
report = json.loads(report_path.read_text())
product = report["target_pixel_products"][0]
fits_path = Path(product["file"])
period = 9.6835  # provisional trial period, days

with fits.open(fits_path, memmap=True) as hdul:
    data = hdul[1].data
    names = {n.upper(): n for n in data.names}
    time = np.asarray(data[names["TIME"]], dtype=float)
    cube = np.asarray(data[names["FLUX"]], dtype=float)
    if "FLUX_BKG" in names:
        cube -= np.asarray(data[names["FLUX_BKG"]], dtype=float)
    quality = np.asarray(data[names["QUALITY"]], dtype=int)
    aperture = np.asarray(hdul[2].data, dtype=np.int64)

aperture_mask = (aperture & 2) > 0
aperture_flux = np.nansum(cube[:, aperture_mask], axis=1)
frame_total = np.nansum(np.clip(cube, 0, None), axis=(1, 2))

good = (
    (quality == 0) & np.isfinite(time) &
    np.isfinite(aperture_flux) & (frame_total > 0)
)
time, cube = time[good], cube[good]
aperture_flux, frame_total = aperture_flux[good], frame_total[good]

phase = ((time - time[0] + period / 2) % period) / period - 0.5
edges = np.linspace(-0.5, 0.5, 101)
centers = (edges[:-1] + edges[1:]) / 2
binned = np.array([
    np.nanmedian(aperture_flux[(phase >= edges[i]) & (phase < edges[i + 1])])
    for i in range(len(edges) - 1)
])
dip_phase = centers[np.nanargmin(binned)]
distance = np.abs((phase - dip_phase + 0.5) % 1 - 0.5)
in_dip = distance <= 0.04
out_dip = distance >= 0.12

# Compare each pixel's share of total cutout flux during and outside the dip.
pixel_fraction = cube / frame_total[:, None, None]
difference = (
    np.nanmedian(pixel_fraction[in_dip], axis=0)
    - np.nanmedian(pixel_fraction[out_dip], axis=0)
) * 100  # percentage points

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(phase, aperture_flux / np.nanmedian(aperture_flux),
                s=2, alpha=0.35)
axes[0].axvline(dip_phase, color="red", linestyle="--")
axes[0].set(xlabel="Phase at 9.6835-day trial period",
            ylabel="Normalized aperture flux",
            title="Folded target-aperture light curve")

limit = np.nanpercentile(np.abs(difference), 98)
im = axes[1].imshow(difference, origin="lower", cmap="coolwarm",
                    vmin=-limit, vmax=limit)
for i, neighbor in enumerate(report["gaia_query"]["records"]):
    x, y = neighbor.get("RA_ICRS"), neighbor.get("DE_ICRS")
    # Use the corrected audit's WCS pixel positions:
    match = next((n for n in product["gaia_neighbors_within_30_arcsec"]
                  if n.get("source_id") == neighbor.get("Source")), None)
    if match:
        axes[1].scatter(match["x_pixel"], match["y_pixel"],
                        marker="+", s=100,
                        label="Gaia target" if i == 0 else "Nearby Gaia source")
axes[1].set(title="Pixel flux-share change: dip minus baseline",
            xlabel="Pixel column", ylabel="Pixel row")
axes[1].legend()
fig.colorbar(im, ax=axes[1], label="Change (percentage points)")
fig.tight_layout()
fig.savefig("/kaggle/working/TIC_229137615_pixel_localization.png", dpi=150)
plt.show()

print("Trial dip phase:", round(float(dip_phase), 4))
print("Cadences in dip / baseline:", int(in_dip.sum()), "/", int(out_dip.sum()))
print("Saved: /kaggle/working/TIC_229137615_pixel_localization.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path
import json

report_path = Path("/kaggle/working/cbs-data/pixel_audits/TIC_229137615.json")
report = json.loads(report_path.read_text())
fits_path = Path(report["target_pixel_products"][0]["file"])

period = 9.6835  # still provisional
dip_phase = 0.265  # measured from this same trial fold

with fits.open(fits_path, memmap=True) as hdul:
    data = hdul[1].data
    names = {n.upper(): n for n in data.names}
    time = np.asarray(data[names["TIME"]], dtype=float)
    cube = np.asarray(data[names["FLUX"]], dtype=float)
    if "FLUX_BKG" in names:
        cube = cube - np.asarray(data[names["FLUX_BKG"]], dtype=float)
    quality = np.asarray(data[names["QUALITY"]], dtype=int)
    aperture = np.asarray(hdul[2].data, dtype=np.int64)

mask = (aperture & 2) > 0
weights = np.where(np.isfinite(cube), np.clip(cube, 0, None), 0)
weights *= mask[None, :, :]

ap_flux = weights.sum(axis=(1, 2))
yy, xx = np.indices(mask.shape)
denom = ap_flux
cx = (weights * xx[None, :, :]).sum(axis=(1, 2)) / denom
cy = (weights * yy[None, :, :]).sum(axis=(1, 2)) / denom

good = (quality == 0) & np.isfinite(time) & (ap_flux > 0) & np.isfinite(cx) & np.isfinite(cy)
time, ap_flux, cx, cy = time[good], ap_flux[good], cx[good], cy[good]

phase = ((time - time[0] + period / 2) % period) / period - 0.5
phase_distance = np.abs((phase - dip_phase + 0.5) % 1 - 0.5)
in_dip = phase_distance <= 0.04
baseline = phase_distance >= 0.12
relative_flux = ap_flux / np.nanmedian(ap_flux[baseline])

print("Trial-window depth (%):",
      round(100 * (1 - np.nanmedian(ap_flux[in_dip]) /
                   np.nanmedian(ap_flux[baseline])), 3))
print("Median centroid shift, dip minus baseline (pixels):",
      round(float(np.nanmedian(cx[in_dip]) - np.nanmedian(cx[baseline])), 4),
      round(float(np.nanmedian(cy[in_dip]) - np.nanmedian(cy[baseline])), 4))
print("Flux-centroid correlations (x, y):",
      np.round(np.corrcoef(relative_flux, cx)[0, 1], 3),
      np.round(np.corrcoef(relative_flux, cy)[0, 1], 3))

cycle = np.floor((time - time[0]) / period).astype(int)
print("\nCycle-by-cycle trial-window depths:")
for k in np.unique(cycle):
    event = (cycle == k) & (phase_distance <= 0.04)
    base = (cycle == k) & (phase_distance >= 0.12)
    if event.sum() >= 5 and base.sum() >= 20:
        depth = 100 * (1 - np.nanmedian(ap_flux[event]) / np.nanmedian(ap_flux[base]))
        print(f"cycle {k}: {depth:.3f}% ({event.sum()} dip, {base.sum()} baseline cadences)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, coord, label in [(axes[0], cx, "x centroid (pixels)"),
                         (axes[1], cy, "y centroid (pixels)")]:
    ax.scatter(coord[baseline], relative_flux[baseline], s=3, alpha=0.2, label="baseline")
    ax.scatter(coord[in_dip], relative_flux[in_dip], s=5, alpha=0.35, color="red", label="trial dip")
    ax.set(xlabel=label, ylabel="Relative aperture flux")
    ax.legend()
fig.suptitle("TIC 229137615: flux versus pixel centroid")
fig.tight_layout()
fig.savefig("/kaggle/working/TIC_229137615_centroid_check.png", dpi=150)
plt.show()
print("Saved: /kaggle/working/TIC_229137615_centroid_check.png")

In [ ]:
import sys
import json
from pathlib import Path

data_root = Path("/kaggle/working/cbs-data")
sys.path.insert(0, "/kaggle/working/cbs-project-src/src")

required = [
    data_root / "features.csv",
    data_root / "series.npy",
    data_root / "scores" / "run.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Injection-recovery inputs missing: {missing}")

from cosmic_black_swan.injections import injection_recovery

result = injection_recovery(
    data_root,
    sample_size=200,
    amplitudes_sigma=(1, 2, 4, 8),
    seed=42,
)
print(json.dumps(result, indent=2))

In [ ]:
import sys, json, shutil
from pathlib import Path

# Find the saved Kaggle output containing features.csv and series.npy.
notebook_inputs = Path("/kaggle/input/notebooks/urajkaushik")
saved_roots = [notebook_inputs / "cosmic-swan-4" / "cbs-data"]
if notebook_inputs.exists():
    saved_roots += list(notebook_inputs.glob("*/cbs-data"))

source_root = next(
    (p for p in dict.fromkeys(saved_roots)
     if (p / "features.csv").exists() and (p / "series.npy").exists()),
    None,
)
if source_root is None:
    print("Searched these saved data folders:")
    for p in dict.fromkeys(saved_roots):
        print(" ", p, "| exists:", p.exists())
    raise FileNotFoundError("Could not find saved features.csv and series.npy.")

data_root = Path("/kaggle/working/cbs-data")
data_root.mkdir(parents=True, exist_ok=True)
(data_root / "scores").mkdir(exist_ok=True)

# Copy both from the same saved run so their rows stay aligned.
for filename in ("features.csv", "series.npy"):
    shutil.copy2(source_root / filename, data_root / filename)

saved_run = source_root / "scores" / "run.json"
if saved_run.exists():
    shutil.copy2(saved_run, data_root / "scores" / "run.json")

# Load the project package.
project_paths = [
    Path("/kaggle/working/cbs-project-src/src"),
    notebook_inputs / "cosmic-swan-4" / "cbs-project-src" / "src",
]
for p in reversed(project_paths):
    if p.exists():
        sys.path.insert(0, str(p))

# If run metadata wasn't saved, recreate it from features (quick; no FITS download).
if not (data_root / "scores" / "run.json").exists():
    from cosmic_black_swan.score import score_isolation_forest
    score_isolation_forest(data_root, contamination=0.02, seed=42)

import pandas as pd
import numpy as np

row_count = len(pd.read_csv(data_root / "features.csv"))
series_count = len(np.load(data_root / "series.npy", mmap_mode="r"))
if row_count != series_count:
    raise ValueError(f"Feature/series rows differ: {row_count} vs {series_count}")

from cosmic_black_swan.injections import injection_recovery

result = injection_recovery(
    data_root,
    sample_size=200,
    amplitudes_sigma=(1, 2, 4, 8),
    seed=42,
)
print(json.dumps(result, indent=2))

In [ ]:
import pandas as pd
from pathlib import Path

map_path = Path("/kaggle/working/cbs-data/scores/blindspot_sensitivity_map.csv")
blindspot = pd.read_csv(map_path)

blindspot["recovered_estimate"] = blindspot["recall"] * blindspot["injections"]

summary = (
    blindspot.groupby(["injection_family", "amplitude_sigma"], as_index=False)
    .agg(
        recovered=("recovered_estimate", "sum"),
        injected=("injections", "sum"),
        detector_cells=("injections", "size"),
    )
)
summary["recovery_rate"] = summary["recovered"] / summary["injected"]
summary.to_csv(
    "/kaggle/working/cbs-data/scores/injection_recovery_summary.csv",
    index=False,
)

display(
    summary.pivot(
        index="injection_family",
        columns="amplitude_sigma",
        values="recovery_rate",
    ).round(3)
)
print("Saved weighted recovery summary.")

In [ ]:
from pathlib import Path

report = """# Cosmic Black Swan — TESS Screening Pilot

## Scope and method

The screening pipeline ranked TESS light curves and produced a reviewed candidate shortlist.
The injection benchmark used a source-disjoint split: 2,823 training sources, 941 calibration
sources, and 942 test sources. It injected 3,200 synthetic signals into 200 selected test
sources. The Isolation Forest threshold was frozen at the 99th percentile of calibration
scores.

These recovery rates measure sensitivity to the tested synthetic templates. They are not
probabilities, false-discovery rates, or guarantees of sensitivity to real events.

## Injection recovery rate

| Synthetic signal | 1σ | 2σ | 4σ | 8σ |
|---|---:|---:|---:|---:|
| Eclipse | 0.000 | 0.000 | 0.420 | 0.990 |
| Flare | 0.000 | 0.000 | 0.110 | 0.900 |
| Periodic | 0.005 | 0.870 | 1.000 | 1.000 |
| Step artifact | 0.000 | 0.085 | 0.995 | 1.000 |

The benchmark indicates morphology-dependent sensitivity: this Isolation Forest recovers
strong periodic signals and steps more readily than weak eclipse or flare injections.

## Candidate review: TIC 229137615

- Sector 2; 9.6835-day **trial period**.
- The trial dip recurs across three observed cycles, with measured depths of 0.199%, 0.378%,
  and 0.299%.
- Flux-to-centroid correlations were small (x = 0.028, y = 0.010), with median centroid
  shifts of +0.0021 and -0.0087 pixels. This gives no obvious evidence that simple centroid
  motion explains the dip.
- The corrected TESS aperture mask selects 45 pixels. A nearby faint Gaia source falls in
  the selected aperture, so pixel contamination remains a consideration.
- This is a single-sector, three-cycle review using a period and phase selected from the
  same data. It remains a provisional variable candidate.

## Conclusion

The screening and candidate-vetting pipeline ran on the analyzed sample, and its synthetic
recovery behavior is now measured for the tested templates. No new transient or other
“cosmic black swan” is confirmed by this work. The leading candidate needs independent
sector validation before stronger claims.

The anomaly scores and percentiles are within-run triage rankings, not calibrated
probabilities or measured false-discovery rates.
"""

output = Path("/kaggle/working/cbs-data/cosmic_swan_screening_pilot_report.md")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(report, encoding="utf-8")
print("Saved:", output)
print(report)

In [ ]:
import sys
import json
import shutil
from pathlib import Path

possible_projects = [
    Path("/kaggle/working/cbs-project-src"),
    Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-project-src"),
]
project_root = next(
    (p for p in possible_projects if (p / "src" / "cosmic_black_swan" / "pixel_audit.py").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the saved cbs-project-src folder.")

sys.path.insert(0, str(project_root / "src"))

data_root = Path("/kaggle/working/cbs-data")
features_file = data_root / "features.csv"

# If needed, copy only the small feature table from the attached notebook output.
if not features_file.exists():
    saved_data = Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
    if not (saved_data / "features.csv").exists():
        raise FileNotFoundError("features.csv was not found in the saved notebook output.")
    data_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(saved_data / "features.csv", features_file)

from cosmic_black_swan.pixel_audit import run_pixel_audit

result = run_pixel_audit(data_root, source_id="229137615", sector=2)
print(json.dumps(result, indent=2))

report = json.loads(Path(result["report"]).read_text())
for product in report["target_pixel_products"]:
    print({
        "sector": product["sector"],
        "camera": product["camera"],
        "ccd": product["ccd"],
        "outside_aperture_flux_fraction": product["outside_aperture_positive_flux_fraction_of_cutout"],
        "quality_zero_fraction": product["quality_zero_cadence_fraction"],
        "bright_neighbors_in_aperture": len(product["bright_gaia_neighbors_in_optimal_aperture"]),
    })

In [ ]:
from cosmic_black_swan.pixel_audit import run_pixel_audit

report_name = "TIC_229137615.json"
report_paths = [
    data_root / "pixel_audits" / report_name,
    Path("/kaggle/input/notebooks/urajkaushik/cosmic-swan-4/cbs-data")
    / "pixel_audits" / report_name,
]

report_path = next((p for p in report_paths if p.exists()), None)

if report_path is None:
    result = run_pixel_audit(data_root, source_id="229137615", sector=2)
    report_path = Path(result["report"])
    print("Ran a fresh pixel audit.")
else:
    print("Reusing saved pixel audit:", report_path)

report = json.loads(report_path.read_text())

for product in report["target_pixel_products"]:
    print({
        "sector": product["sector"],
        "camera": product["camera"],
        "ccd": product["ccd"],
        "outside_aperture_flux_fraction":
            product["outside_aperture_positive_flux_fraction_of_cutout"],
        "quality_zero_fraction":
            product["quality_zero_cadence_fraction"],
        "bright_neighbors_in_aperture":
            len(product["bright_gaia_neighbors_in_optimal_aperture"]),
    })